# 📱 ABSA — Phân tích cảm xúc đa khía cạnh cho đánh giá điện thoại
## Tuần 2: Pipeline Preprocessing + Baseline (Rule-based) + Few-shot Prompting (Gemini)

**Nguồn dữ liệu:** Thế Giới Di Động, Điện Máy Xanh, CellPhones  
**Các Aspect:** PIN, CAMERA, HIỆU NĂNG, MÀN HÌNH, THIẾT KẾ, GIÁ CẢ, TỔNG THỂ  
**Mô hình:** Rule-based Baseline vs Few-shot Gemini 1.5 Flash

---
### 📌 Mục lục
1. Cài đặt thư viện
2. Load & khám phá dữ liệu (EDA)
3. Preprocessing Pipeline
4. Baseline Model (Rule-based)
5. Few-shot Prompting với Gemini
6. So sánh kết quả hai phương pháp
7. Lưu kết quả


## ⚙️ 1. Cài đặt thư viện

In [ ]:
# Cài các thư viện cần thiết
!pip install underthesea pyvi pandas numpy matplotlib seaborn scikit-learn google-generativeai tqdm -q

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import time
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# NLP tiếng Việt
from underthesea import word_tokenize, sent_tokenize

# Gemini
import google.generativeai as genai

# Sklearn
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

print('✅ Import thành công!')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

## 📂 2. Load & Khám phá dữ liệu (EDA)

> **Hướng dẫn:** Upload file CSV của nhóm lên Colab, đổi tên biến `CSV_PATH` cho đúng.  
> File CSV cần có ít nhất 2 cột: `review` (văn bản đánh giá) và `rating` (1-5 sao).

In [ ]:
# ============================================================
# 🔧 CẤU HÌNH — Thay đổi theo dữ liệu thực của nhóm
# ============================================================
CSV_PATH = 'reviews.csv'       # Đường dẫn file CSV
REVIEW_COL = 'review'          # Tên cột chứa nội dung đánh giá
RATING_COL = 'rating'          # Tên cột chứa số sao (1-5)
SOURCE_COL = 'source'          # Tên cột nguồn (nếu có)
PRODUCT_COL = 'product'        # Tên cột tên sản phẩm (nếu có)

# ============================================================
# Nếu chưa có file thực, dùng dữ liệu mẫu để chạy thử
# ============================================================
SAMPLE_DATA = [
    {"review": "Pin điện thoại này dùng được cả ngày, camera chụp rất đẹp nét, màn hình sắc sảo", "rating": 5, "source": "thegioididong", "product": "Samsung S24"},
    {"review": "Máy lag hay bị giật, pin hao quá chỉ được nửa ngày là hết, giá lại đắt", "rating": 2, "source": "dienmayxanh", "product": "iPhone 15"},
    {"review": "Thiết kế đẹp mỏng nhẹ cầm thoải mái, nhưng camera selfie chụp hơi xấu", "rating": 3, "source": "cellphones", "product": "Xiaomi 14"},
    {"review": "Hiệu năng mạnh chạy game mượt mà, màn hình 120Hz rất đẹp, pin sạc nhanh 65W", "rating": 5, "source": "thegioididong", "product": "OPPO Find X7"},
    {"review": "Giá quá cao so với cấu hình, màn hình tối không xem được ngoài trời", "rating": 2, "source": "dienmayxanh", "product": "Pixel 8"},
    {"review": "Camera quay video 4K rất mượt, nhưng máy nóng khi chơi game lâu", "rating": 3, "source": "cellphones", "product": "Samsung A55"},
    {"review": "Sản phẩm bình thường không có gì nổi bật, mua về dùng tạm", "rating": 3, "source": "thegioididong", "product": "Vivo Y36"},
    {"review": "Pin trâu lắm, dùng 2 ngày mới cần sạc. Camera đêm chụp tối nhưng vẫn rõ", "rating": 4, "source": "cellphones", "product": "Realme 12"},
    {"review": "Thiết kế sang trọng, màn hình AMOLED cực đẹp, giá hơi đắt nhưng xứng đáng", "rating": 5, "source": "dienmayxanh", "product": "Samsung S23"},
    {"review": "Máy bị lag sau 3 tháng dùng, pin xuống cấp nhanh, không đáng tiền", "rating": 1, "source": "thegioididong", "product": "Redmi Note 13"},
    {"review": "Camera chụp ảnh sắc nét màu sắc trung thực, app ảnh nhiều tính năng hay", "rating": 4, "source": "cellphones", "product": "iPhone 15 Pro"},
    {"review": "Hiệu năng tốt pin ổn, giá tầm trung hợp lý cho cấu hình này", "rating": 4, "source": "dienmayxanh", "product": "Samsung A35"},
]

try:
    df = pd.read_csv(CSV_PATH)
    print(f'✅ Đã load file CSV: {len(df)} reviews')
except FileNotFoundError:
    print('⚠️  Không tìm thấy file CSV — Dùng dữ liệu mẫu để demo')
    df = pd.DataFrame(SAMPLE_DATA)

print(f'\n📊 Shape: {df.shape}')
print(f'📋 Columns: {list(df.columns)}')
df.head()

In [ ]:
# ============================================================
# EDA — Thống kê & Visualize
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('📊 Exploratory Data Analysis — Reviews điện thoại', fontsize=14, fontweight='bold')

# 1. Phân phối Rating
rating_counts = df[RATING_COL].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
axes[0].bar(rating_counts.index, rating_counts.values, color=colors[:len(rating_counts)])
axes[0].set_title('Phân phối Rating (số sao)')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Số lượng')
for i, v in enumerate(rating_counts.values):
    axes[0].text(rating_counts.index[i], v + 0.5, str(v), ha='center', fontweight='bold')

# 2. Phân phối nguồn dữ liệu (nếu có)
if SOURCE_COL in df.columns:
    source_counts = df[SOURCE_COL].value_counts()
    axes[1].pie(source_counts.values, labels=source_counts.index,
                autopct='%1.1f%%', colors=['#3498db','#e74c3c','#2ecc71','#9b59b6'])
    axes[1].set_title('Nguồn dữ liệu')
else:
    axes[1].text(0.5, 0.5, 'Không có\ncột source', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Nguồn dữ liệu')

# 3. Độ dài review
df['review_len'] = df[REVIEW_COL].astype(str).apply(len)
axes[2].hist(df['review_len'], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[2].set_title('Độ dài Review (ký tự)')
axes[2].set_xlabel('Số ký tự'); axes[2].set_ylabel('Tần suất')
axes[2].axvline(df['review_len'].mean(), color='red', linestyle='--', label=f"Mean: {df['review_len'].mean():.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight')
plt.show()

print(f"\n📈 Thống kê cơ bản:")
print(f"   Tổng số reviews   : {len(df)}")
print(f"   Rating trung bình : {df[RATING_COL].mean():.2f} ⭐")
print(f"   Độ dài TB (ký tự) : {df['review_len'].mean():.0f}")
print(f"   Review ngắn nhất  : {df['review_len'].min()} ký tự")
print(f"   Review dài nhất   : {df['review_len'].max()} ký tự")

## 🧹 3. Preprocessing Pipeline

In [ ]:
# ============================================================
# Từ điển viết tắt & teencode tiếng Việt phổ biến
# ============================================================
TEENCODE_DICT = {
    'ok': 'ổn', 'oke': 'ổn', 'okela': 'ổn', 'oce': 'ổn',
    'good': 'tốt', 'gud': 'tốt', 'goodd': 'tốt',
    'bad': 'tệ', 'badd': 'tệ',
    'nice': 'đẹp', 'dep': 'đẹp',
    'kb': 'không biết', 'k': 'không', 'ko': 'không', 'kh': 'không',
    'đc': 'được', 'dc': 'được', 'dk': 'được',
    'vs': 'với', 'voi': 'với',
    'cx': 'cũng', 'cg': 'cũng',
    'bt': 'bình thường', 'bth': 'bình thường',
    'sp': 'sản phẩm', 'sp': 'sản phẩm',
    'nv': 'nhân viên',
    'ship': 'giao hàng', 'shiip': 'giao hàng',
    'kém': 'kém', 'kem': 'kém',
    'lag': 'giật lag', 'giật': 'giật lag',
    'gg': 'google', 'fb': 'facebook',
    'màn': 'màn hình', 'pin': 'pin', 'cam': 'camera',
    'hsd': 'hiệu suất', 'hd': 'hiệu năng',
    'nhanh': 'nhanh', 'cham': 'chậm', 'chậm': 'chậm',
    ':)': 'vui', ':D': 'vui', ':(': 'buồn', ':((': 'buồn',
    '👍': 'tốt', '👎': 'tệ', '❤️': 'yêu thích', '💯': 'hoàn hảo',
}

# ============================================================
# Hàm preprocessing
# ============================================================
def remove_html_tags(text):
    """Xóa HTML tags"""
    return re.sub(r'<[^>]+>', '', str(text))

def normalize_unicode(text):
    """Chuẩn hóa unicode tiếng Việt"""
    # Chuẩn hóa các ký tự đặc biệt
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)  # Zero-width chars
    return text

def expand_teencode(text):
    """Thay thế teencode/viết tắt"""
    words = text.split()
    return ' '.join([TEENCODE_DICT.get(w.lower(), w) for w in words])

def normalize_whitespace(text):
    """Chuẩn hóa khoảng trắng"""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def normalize_repeated_chars(text):
    """đẹpppp → đẹp"""
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def lowercase(text):
    return text.lower()

def word_segment_vi(text):
    """Tách từ tiếng Việt bằng underthesea"""
    try:
        return word_tokenize(text, format='text')
    except:
        return text

def full_preprocess(text, do_word_segment=False):
    """Pipeline đầy đủ"""
    text = str(text)
    text = remove_html_tags(text)
    text = normalize_unicode(text)
    text = lowercase(text)
    text = normalize_repeated_chars(text)
    text = expand_teencode(text)
    text = normalize_whitespace(text)
    if do_word_segment:
        text = word_segment_vi(text)
    return text

# ============================================================
# Áp dụng lên toàn bộ dataset
# ============================================================
print('🔄 Đang preprocessing...')
df['review_clean'] = df[REVIEW_COL].apply(lambda x: full_preprocess(x, do_word_segment=False))
df['review_segmented'] = df[REVIEW_COL].apply(lambda x: full_preprocess(x, do_word_segment=True))

print('\n📋 Ví dụ kết quả preprocessing:')
print('─' * 80)
for i in range(min(3, len(df))):
    print(f"[{i+1}] Gốc    : {df[REVIEW_COL].iloc[i]}")
    print(f"    Cleaned: {df['review_clean'].iloc[i]}")
    print(f"    Segment: {df['review_segmented'].iloc[i]}")
    print('─' * 80)

print(f'\n✅ Preprocessing xong: {len(df)} reviews')

## 🔖 4. Baseline Model — Rule-based Aspect Sentiment

Phương pháp: Dùng từ điển keyword để xác định Aspect, sau đó đếm từ tích cực/tiêu cực để xác định Sentiment.

In [ ]:
# ============================================================
# Từ điển Aspect Keywords (cho điện thoại)
# ============================================================
ASPECT_KEYWORDS = {
    'PIN': [
        'pin', 'battery', 'sạc', 'trâu', 'hao pin', 'lâu hết', 'nhanh hết',
        'sạc nhanh', 'dung lượng pin', 'pin lâu', 'pin yếu', 'hết pin', 'pin chết'
    ],
    'CAMERA': [
        'camera', 'chụp', 'ảnh', 'selfie', 'quay', 'video', 'zoom', 'xóa phông',
        'chụp đêm', 'chụp tối', 'đèn flash', 'resolution', 'megapixel', 'hình ảnh'
    ],
    'HIỆU NĂNG': [
        'nhanh', 'chậm', 'lag', 'giật', 'mượt', 'chip', 'ram', 'hiệu năng',
        'hiệu suất', 'xử lý', 'tốc độ', 'game', 'đa nhiệm', 'snapdragon',
        'dimensity', 'bionic', 'helio', 'nóng máy', 'máy nóng'
    ],
    'MÀN HÌNH': [
        'màn hình', 'màn', 'screen', 'display', 'amoled', 'oled', 'lcd',
        'sắc nét', 'độ sáng', 'tối', 'sáng', 'tần số quét', '120hz', '90hz',
        'hiển thị', 'màu sắc', 'tương phản'
    ],
    'THIẾT KẾ': [
        'thiết kế', 'đẹp', 'xấu', 'mỏng', 'nặng', 'nhẹ', 'cầm', 'vỏ',
        'sang trọng', 'cao cấp', 'nhựa', 'kính', 'nhôm', 'titan',
        'ngoại hình', 'hình dáng', 'màu sắc máy', 'kích thước'
    ],
    'GIÁ CẢ': [
        'giá', 'tiền', 'đắt', 'rẻ', 'xứng', 'hợp lý', 'cao', 'thấp',
        'worth', 'đáng tiền', 'không đáng', 'tầm tiền', 'tầm giá', 'chi phí'
    ],
}

# ============================================================
# Từ điển Sentiment
# ============================================================
POSITIVE_WORDS = [
    'tốt', 'đẹp', 'tuyệt', 'xuất sắc', 'hoàn hảo', 'thích', 'yêu', 'ổn',
    'mượt', 'nhanh', 'sắc nét', 'rõ', 'trâu', 'lâu', 'mạnh', 'ấn tượng',
    'hài lòng', 'xứng đáng', 'hợp lý', 'rẻ', 'giá tốt', 'đáng tiền',
    'sang trọng', 'cao cấp', 'mỏng', 'nhẹ', 'đẹp trai', 'lung linh',
    'recommend', 'perfect', 'good', 'nice', 'great', 'awesome',
    'sắc', 'nét', 'tươi', 'chuẩn', 'chính xác', 'chi tiết', '👍', '💯'
]

NEGATIVE_WORDS = [
    'tệ', 'xấu', 'kém', 'thất vọng', 'không tốt', 'lag', 'giật', 'chậm',
    'nóng', 'hao', 'yếu', 'đắt', 'không đáng', 'tối', 'mờ', 'nhiễu',
    'lỗi', 'hỏng', 'vỡ', 'xước', 'trầy', 'nhanh hết', 'mau hết',
    'plastic', 'rẻ tiền', 'không xứng', 'bình thường', 'không nổi bật',
    'bad', 'poor', 'terrible', 'awful', '👎', ':('
]

NEGATION_WORDS = ['không', 'chưa', 'chẳng', 'chả', 'chưa hề', 'không hề', 'cũng không']

print('✅ Từ điển đã sẵn sàng!')
print(f'   Số aspect  : {len(ASPECT_KEYWORDS)}')
print(f'   Positive   : {len(POSITIVE_WORDS)} từ')
print(f'   Negative   : {len(NEGATIVE_WORDS)} từ')

In [ ]:
def detect_aspects(text):
    """Xác định aspect xuất hiện trong review"""
    detected = []
    text_lower = text.lower()
    for aspect, keywords in ASPECT_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            detected.append(aspect)
    return detected if detected else ['TỔNG THỂ']


def get_sentiment_score(text):
    """Tính điểm sentiment có tính đến phủ định"""
    words = text.lower().split()
    pos_score = 0
    neg_score = 0
    
    for i, word in enumerate(words):
        # Kiểm tra phủ định trước từ
        has_negation = any(
            neg in ' '.join(words[max(0, i-2):i])
            for neg in NEGATION_WORDS
        )
        
        if word in POSITIVE_WORDS:
            if has_negation:
                neg_score += 1
            else:
                pos_score += 1
        elif word in NEGATIVE_WORDS:
            if has_negation:
                pos_score += 0.5
            else:
                neg_score += 1
    
    return pos_score, neg_score


def classify_sentiment_from_score(pos, neg):
    """Phân loại sentiment từ điểm số"""
    if pos == 0 and neg == 0:
        return 'neutral'
    if pos > neg:
        return 'positive'
    elif neg > pos:
        return 'negative'
    else:
        return 'neutral'


def rule_based_absa(text):
    """Phân tích ABSA bằng rule-based"""
    aspects = detect_aspects(text)
    pos, neg = get_sentiment_score(text)
    overall_sentiment = classify_sentiment_from_score(pos, neg)
    
    results = {}
    for aspect in aspects:
        results[aspect] = overall_sentiment
    
    return {
        'aspects': aspects,
        'sentiments': results,
        'pos_score': pos,
        'neg_score': neg,
        'overall': overall_sentiment
    }


# ============================================================
# Test thử
# ============================================================
test_reviews = [
    "Pin điện thoại này dùng được cả ngày, camera chụp rất đẹp nét",
    "Máy lag hay bị giật, pin hao quá chỉ được nửa ngày là hết",
    "Thiết kế đẹp mỏng nhẹ cầm thoải mái, nhưng camera selfie không đẹp",
]

print('🧪 Test Rule-based ABSA:')
print('=' * 70)
for r in test_reviews:
    result = rule_based_absa(r)
    print(f'\n📝 Review: {r}')
    print(f'   Aspects   : {result["aspects"]}')
    print(f'   Sentiments: {result["sentiments"]}')
    print(f'   (+) {result["pos_score"]} / (-) {result["neg_score"]} → Overall: {result["overall"]}')

In [ ]:
# ============================================================
# Áp dụng lên toàn bộ dataset
# ============================================================
print('🔄 Chạy Rule-based ABSA trên toàn bộ dataset...')
results_baseline = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    res = rule_based_absa(row['review_clean'])
    res['review'] = row['review_clean']
    res['rating'] = row[RATING_COL]
    results_baseline.append(res)

df_baseline = pd.DataFrame(results_baseline)

# ============================================================
# Tính accuracy bằng cách so sánh với rating
# rating 4-5 → positive, 3 → neutral, 1-2 → negative
# ============================================================
def rating_to_sentiment(rating):
    if rating >= 4: return 'positive'
    elif rating == 3: return 'neutral'
    else: return 'negative'

df_baseline['true_sentiment'] = df_baseline['rating'].apply(rating_to_sentiment)
df_baseline['pred_sentiment'] = df_baseline['overall']

print('\n📊 Kết quả Baseline (Rule-based):')
print('─' * 50)
print(classification_report(
    df_baseline['true_sentiment'],
    df_baseline['pred_sentiment'],
    target_names=['negative', 'neutral', 'positive'],
    zero_division=0
))

f1_macro = f1_score(df_baseline['true_sentiment'], df_baseline['pred_sentiment'],
                    average='macro', zero_division=0)
print(f'🎯 F1-score Macro (Baseline): {f1_macro:.4f}')

# Phân phối aspect
all_aspects = [a for aspects in df_baseline['aspects'] for a in aspects]
aspect_counts = Counter(all_aspects)
print('\n📌 Phân phối Aspect được phát hiện:')
for asp, cnt in sorted(aspect_counts.items(), key=lambda x: -x[1]):
    bar = '█' * int(cnt / max(aspect_counts.values()) * 20)
    print(f'   {asp:<15} {bar} {cnt}')

## 🤖 5. Few-shot Prompting với Gemini 1.5 Flash

> **Lấy API key miễn phí tại:** https://aistudio.google.com/app/apikey

In [ ]:
# ============================================================
# Cấu hình Gemini API
# ============================================================
from google.colab import userdata

# Cách 1: Dùng Colab Secrets (khuyến nghị)
# Vào biểu tượng 🔑 bên trái → Thêm secret tên GEMINI_API_KEY
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ Đã lấy API key từ Colab Secrets')
except:
    # Cách 2: Nhập thủ công (không khuyến nghị để public)
    GEMINI_API_KEY = ''  # ← Dán API key vào đây nếu dùng cách 2
    if not GEMINI_API_KEY:
        print('⚠️  Chưa có API key — Sẽ dùng mock response để demo')

USE_MOCK = not bool(GEMINI_API_KEY)

if not USE_MOCK:
    genai.configure(api_key=GEMINI_API_KEY)
    model_gemini = genai.GenerativeModel('gemini-1.5-flash')
    print('✅ Gemini 1.5 Flash đã sẵn sàng!')

In [ ]:
# ============================================================
# Few-shot Prompt Template
# ============================================================
FEW_SHOT_EXAMPLES = [
    {
        "review": "Pin điện thoại này dùng cả ngày thoải mái, camera chụp ảnh rất đẹp nét",
        "output": {"PIN": "positive", "CAMERA": "positive"}
    },
    {
        "review": "Máy bị lag hay giật, pin hao quá, giá lại đắt không xứng",
        "output": {"HIỆU NĂNG": "negative", "PIN": "negative", "GIÁ CẢ": "negative"}
    },
    {
        "review": "Thiết kế đẹp mỏng nhẹ, nhưng camera selfie không được tốt lắm",
        "output": {"THIẾT KẾ": "positive", "CAMERA": "negative"}
    },
    {
        "review": "Màn hình 120Hz cực mượt, hiệu năng chip mạnh chạy game ổn",
        "output": {"MÀN HÌNH": "positive", "HIỆU NĂNG": "positive"}
    },
    {
        "review": "Giá hợp lý cho tầm tiền này, dùng bình thường không có gì đặc biệt",
        "output": {"GIÁ CẢ": "positive", "TỔNG THỂ": "neutral"}
    },
]

SYSTEM_PROMPT = """Bạn là chuyên gia phân tích cảm xúc đa khía cạnh (ABSA) cho đánh giá điện thoại tiếng Việt.

Các khía cạnh (aspect) cần phân tích:
- PIN: pin, sạc, dung lượng pin
- CAMERA: camera, chụp ảnh, quay video
- HIỆU NĂNG: tốc độ, chip, RAM, game, lag
- MÀN HÌNH: màn hình, độ sáng, màu sắc hiển thị
- THIẾT KẾ: ngoại hình, kích thước, chất liệu
- GIÁ CẢ: giá tiền, có đáng tiền không
- TỔNG THỂ: nhận xét chung không thuộc aspect nào

Nhãn sentiment: "positive", "negative", "neutral"

QUY TẮC:
1. Chỉ trả về các aspect được đề cập trong review
2. Nếu không có aspect rõ ràng, dùng TỔNG THỂ
3. Trả về JSON hợp lệ, KHÔNG giải thích thêm
4. Format: {"ASPECT": "sentiment", ...}"""

def build_few_shot_prompt(review_text):
    """Xây dựng prompt few-shot"""
    examples_str = "\n".join([
        f'Review: "{ex["review"]}"\nOutput: {json.dumps(ex["output"], ensure_ascii=False)}'
        for ex in FEW_SHOT_EXAMPLES
    ])
    
    prompt = f"""{SYSTEM_PROMPT}

=== VÍ DỤ FEW-SHOT ===
{examples_str}

=== PHÂN TÍCH ===
Review: "{review_text}"
Output:"""
    return prompt

print('✅ Prompt template đã sẵn sàng!')
print('\n📋 Ví dụ prompt cho review đầu tiên:')
print('─' * 60)
print(build_few_shot_prompt(test_reviews[0])[:500] + '...')

In [ ]:
# ============================================================
# Hàm gọi Gemini API
# ============================================================
def parse_gemini_response(response_text):
    """Parse JSON từ response của Gemini"""
    try:
        # Tìm JSON trong response
        match = re.search(r'\{[^}]+\}', response_text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return json.loads(response_text.strip())
    except:
        return {'TỔNG THỂ': 'neutral'}  # fallback


MOCK_RESPONSES = [
    {"PIN": "positive", "CAMERA": "positive"},
    {"HIỆU NĂNG": "negative", "PIN": "negative", "GIÁ CẢ": "negative"},
    {"THIẾT KẾ": "positive", "CAMERA": "negative"},
    {"MÀN HÌNH": "positive", "HIỆU NĂNG": "positive"},
    {"GIÁ CẢ": "positive", "TỔNG THỂ": "neutral"},
    {"CAMERA": "positive", "HIỆU NĂNG": "negative"},
    {"TỔNG THỂ": "neutral"},
    {"PIN": "positive", "CAMERA": "positive"},
    {"THIẾT KẾ": "positive", "MÀN HÌNH": "positive", "GIÁ CẢ": "neutral"},
    {"HIỆU NĂNG": "negative", "PIN": "negative"},
    {"CAMERA": "positive"},
    {"HIỆU NĂNG": "positive", "GIÁ CẢ": "positive"},
]

def gemini_absa(review_text, idx=0):
    """Gọi Gemini để phân tích ABSA"""
    if USE_MOCK:
        # Mock response khi chưa có API key
        return MOCK_RESPONSES[idx % len(MOCK_RESPONSES)]
    
    try:
        prompt = build_few_shot_prompt(review_text)
        response = model_gemini.generate_content(prompt)
        result = parse_gemini_response(response.text)
        time.sleep(0.5)  # Tránh rate limit
        return result
    except Exception as e:
        print(f'⚠️  Lỗi API: {e}')
        return {'TỔNG THỂ': 'neutral'}


# ============================================================
# Test Few-shot
# ============================================================
print('🧪 Test Few-shot Gemini ABSA:')
print('=' * 70)
for i, r in enumerate(test_reviews):
    result = gemini_absa(r, i)
    print(f'\n📝 Review: {r}')
    for aspect, sentiment in result.items():
        emoji = '✅' if sentiment == 'positive' else ('❌' if sentiment == 'negative' else '➖')
        print(f'   {emoji} {aspect}: {sentiment}')

if USE_MOCK:
    print('\n⚠️  Đang dùng Mock response — Thêm GEMINI_API_KEY để dùng thật')

In [ ]:
# ============================================================
# Chạy Few-shot trên toàn bộ dataset
# ============================================================
print('🔄 Chạy Few-shot Gemini trên toàn bộ dataset...')
print('   (Dùng mock response nếu chưa có API key)\n')

results_gemini = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    pred = gemini_absa(row['review_clean'], idx)
    
    # Tính overall sentiment
    sentiments = list(pred.values())
    pos_count = sentiments.count('positive')
    neg_count = sentiments.count('negative')
    if pos_count > neg_count:
        overall = 'positive'
    elif neg_count > pos_count:
        overall = 'negative'
    else:
        overall = 'neutral'
    
    results_gemini.append({
        'review': row['review_clean'],
        'rating': row[RATING_COL],
        'aspects': list(pred.keys()),
        'sentiments': pred,
        'overall': overall
    })

df_gemini = pd.DataFrame(results_gemini)
df_gemini['true_sentiment'] = df_gemini['rating'].apply(rating_to_sentiment)
df_gemini['pred_sentiment'] = df_gemini['overall']

print('\n📊 Kết quả Few-shot Gemini:')
print('─' * 50)
print(classification_report(
    df_gemini['true_sentiment'],
    df_gemini['pred_sentiment'],
    target_names=['negative', 'neutral', 'positive'],
    zero_division=0
))

f1_gemini = f1_score(df_gemini['true_sentiment'], df_gemini['pred_sentiment'],
                     average='macro', zero_division=0)
print(f'🎯 F1-score Macro (Few-shot Gemini): {f1_gemini:.4f}')

## 📊 6. So sánh kết quả hai phương pháp

In [ ]:
# ============================================================
# Bảng so sánh
# ============================================================
from sklearn.metrics import accuracy_score

acc_baseline = accuracy_score(df_baseline['true_sentiment'], df_baseline['pred_sentiment'])
acc_gemini   = accuracy_score(df_gemini['true_sentiment'],   df_gemini['pred_sentiment'])
f1_baseline_val = f1_score(df_baseline['true_sentiment'], df_baseline['pred_sentiment'],
                            average='macro', zero_division=0)
f1_gemini_val   = f1_score(df_gemini['true_sentiment'], df_gemini['pred_sentiment'],
                            average='macro', zero_division=0)

comparison = pd.DataFrame({
    'Phương pháp': ['Rule-based Baseline', 'Few-shot Gemini'],
    'Accuracy': [f'{acc_baseline:.4f}', f'{acc_gemini:.4f}'],
    'F1-Macro': [f'{f1_baseline_val:.4f}', f'{f1_gemini_val:.4f}'],
    'Ưu điểm': ['Nhanh, không cần API, dễ giải thích', 'Hiểu ngữ cảnh, tiếng Việt tốt'],
    'Nhược điểm': ['Không hiểu ngữ cảnh, phụ thuộc từ điển', 'Phụ thuộc API, chậm hơn'],
})

print('\n📋 BẢNG SO SÁNH HAI PHƯƠNG PHÁP')
print('=' * 70)
print(comparison.to_string(index=False))

# ============================================================
# Visualize
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('So sánh Rule-based vs Few-shot Gemini', fontsize=14, fontweight='bold')

methods = ['Rule-based\nBaseline', 'Few-shot\nGemini']
colors_bar = ['#3498db', '#e74c3c']

# Accuracy
bars1 = axes[0].bar(methods, [acc_baseline, acc_gemini], color=colors_bar, width=0.4)
axes[0].set_title('Accuracy'); axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Score')
for bar, val in zip(bars1, [acc_baseline, acc_gemini]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontweight='bold')

# F1-Macro
bars2 = axes[1].bar(methods, [f1_baseline_val, f1_gemini_val], color=colors_bar, width=0.4)
axes[1].set_title('F1-score Macro'); axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Score')
for bar, val in zip(bars2, [f1_baseline_val, f1_gemini_val]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

winner = 'Few-shot Gemini' if f1_gemini_val > f1_baseline_val else 'Rule-based Baseline'
print(f'\n🏆 Phương pháp tốt hơn (F1): {winner}')

In [ ]:
# ============================================================
# Phân tích lỗi — Confusion Matrix
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix', fontsize=14, fontweight='bold')

labels = ['negative', 'neutral', 'positive']

for ax, (title, true_col, pred_col, df_r) in zip(axes, [
    ('Rule-based Baseline', 'true_sentiment', 'pred_sentiment', df_baseline),
    ('Few-shot Gemini',     'true_sentiment', 'pred_sentiment', df_gemini),
]):
    cm = confusion_matrix(df_r[true_col], df_r[pred_col], labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels)
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()
print('✅ Confusion matrix đã lưu!')

## 💾 7. Lưu kết quả

In [ ]:
# ============================================================
# Lưu CSV kết quả
# ============================================================
df_output = df[['review_clean', RATING_COL]].copy()
df_output['true_sentiment'] = df_output[RATING_COL].apply(rating_to_sentiment)
df_output['baseline_sentiment']   = df_baseline['pred_sentiment'].values
df_output['baseline_aspects']     = df_baseline['aspects'].apply(str).values
df_output['gemini_sentiment']     = df_gemini['pred_sentiment'].values
df_output['gemini_aspects']       = df_gemini['aspects'].apply(str).values

df_output.to_csv('absa_results_week2.csv', index=False, encoding='utf-8-sig')
print('✅ Đã lưu kết quả vào: absa_results_week2.csv')
print(f'   Tổng số reviews: {len(df_output)}')

# ============================================================
# Tóm tắt cuối
# ============================================================
print('\n' + '=' * 60)
print('📋 TÓM TẮT KẾT QUẢ TUẦN 2')
print('=' * 60)
print(f'  Tổng reviews phân tích  : {len(df)}')
print(f'  Số aspect được định nghĩa: {len(ASPECT_KEYWORDS) + 1} (gồm TỔNG THỂ)')
print(f'  Rule-based — Accuracy   : {acc_baseline:.4f} | F1: {f1_baseline_val:.4f}')
print(f'  Few-shot Gemini — Acc   : {acc_gemini:.4f} | F1: {f1_gemini_val:.4f}')
print('=' * 60)
print('\n📁 Files đã tạo:')
print('   - absa_results_week2.csv')
print('   - eda_overview.png')
print('   - model_comparison.png')
print('   - confusion_matrix.png')
print('\n🚀 Kế hoạch Tuần 3: Fine-tuning PhoBERT + Kết nối Web UI')